# CSE 151B — Submission notebook (`private.jsonl`)

Live in **`notebooks/submission.ipynb`**. Same inference stack as [`full_public.ipynb`](full_public.ipynb), run on all of **`data/private.jsonl`** (no labels).

1. (**Colab A100**) `%pip` → restart runtime → Drive copy cell.
2. Set **`MAX_TOKENS`** in §2 (default **32k**, matching pub-003).
3. Prompts are **per question**: baseline for MCQ and single-blank free-form; multi-blank format when `[ANS]` count > 1.
4. Generate with checkpointing → write **`results/submission.csv`**.

**Output:** CSV with columns `id`, `response` (full model trace, CSV-escaped). Checkpoint: `results/submission.responses.jsonl`.

### Google Colab

**Colab (A100):** run the `%pip` cell below, restart runtime, continue. **Local:** use your venv — same packages (`vllm`, `transformers`, …).

> **Note:** `bitsandbytes` is not needed — Qwen3-4B fits in native bf16 on A100 (~8 GB), which is faster than quantized loading.

In [1]:
!git clone https://github.com/keguangzhang/151B_SP26_Competition.git
%cd 151B_SP26_Competition
!git checkout amy-branch

Cloning into '151B_SP26_Competition'...
remote: Enumerating objects: 498, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 498 (delta 57), reused 65 (delta 35), pack-reused 384 (from 2)
Receiving objects: 100% (498/498), 15.59 MiB | 35.56 MiB/s, done.
Resolving deltas: 100% (227/227), done.
/content/151B_SP26_Competition
Branch 'amy-branch' set up to track remote branch 'amy-branch' from 'origin'.
Switched to a new branch 'amy-branch'


In [2]:
# Colab: skip when working locally with an existing venv.
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 9.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 MB 134.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 116.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 410.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 217.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 416.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 287.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 283.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.1/338.1 MB 

## 2. Imports & configuration

- `MAX_TOKENS` — **32768** = pub-003 path (default)
- `SUBMISSION_CSV` — `results/submission.csv`
- `DATA_PATH` — `data/private.jsonl`

Prompts are chosen automatically in §5 (no variant knob).

In [ ]:
!nvidia-smi

Sun May 31 02:20:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import csv
import json
import os
import re
from collections import Counter
from fractions import Fraction
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()

MAX_TOKENS = 12000

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"
DATA_PATH = str(REPO_ROOT / "/content/151B_SP26_Competition/data/public.jsonl")
SUBMISSION_CSV = str(REPO_ROOT / "/content/151B_SP26_Competition/results/public_4_result.csv")

# Self-consistency settings
SELF_CONSISTENCY_K = 4

# Only run verifier when vote agreement is weak or output format looks bad.
USE_VERIFIER = False
VERIFY_AGREEMENT_THRESHOLD = 0.67

SUBMIT_FINAL_ONLY = False

_TOKEN_K = MAX_TOKENS // 1024

print(f"REPO_ROOT      = {REPO_ROOT}")
print(f"MAX_TOKENS     = {MAX_TOKENS} ({_TOKEN_K}k)")
print(f"SELF_CONSISTENCY_K = {SELF_CONSISTENCY_K}")
print(f"USE_VERIFIER   = {USE_VERIFIER}")
print(f"SUBMIT_FINAL_ONLY = {SUBMIT_FINAL_ONLY}")
print(f"SUBMISSION_CSV = {SUBMISSION_CSV}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import sys
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

REPO_ROOT      = /content/151B_SP26_Competition
MAX_TOKENS     = 12000 (11k)
SELF_CONSISTENCY_K = 4
USE_VERIFIER   = False
SUBMIT_FINAL_ONLY = False
SUBMISSION_CSV = /content/151B_SP26_Competition/results/public_4_result.csv


In [ ]:
%pip install "transformers==4.55.4"

## 3. Colab: copy `private.jsonl` from Drive

Upload **`private.jsonl`** to `My Drive/CSE151B/data/private.jsonl` (or change `DRIVE_PRIVATE`). Skip on local clone when `data/private.jsonl` exists.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use repo `data/public.jsonl`.")
else:
    drive.mount("/content/drive")
    DRIVE_PRIVATE = Path("/CSE151B/data/public.jsonl")
    DRIVE_PROJECT_ROOT = DRIVE_PRIVATE.parent.parent
    if not DRIVE_PRIVATE.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_PRIVATE}\nUpload private.jsonl or set DRIVE_PRIVATE."
        )
    (REPO_ROOT / "data").mkdir(parents=True, exist_ok=True)
    dest = REPO_ROOT / "data/public.jsonl"
    shutil.copy2(DRIVE_PRIVATE, dest)
    print(f"Copied to {dest.resolve()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: Missing on Drive: /CSE151B/data/public.jsonl
Upload private.jsonl or set DRIVE_PRIVATE.

## 4. Load dataset

Private rows have `question`, optional `options`, and `id` — **no** `answer` field.

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices — present for **MCQ**, absent for **free-form** |

In [4]:
def n_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")


data = [json.loads(line) for line in open(DATA_PATH)]


n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form) from {DATA_PATH}")

mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))
multi_sample = next(
    (d for d in data if not d.get("options") and n_ans_placeholders(d["question"]) > 1),
    free_sample,
)

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2)[:1200], "...\n" if len(json.dumps(mcq_sample)) > 1200 else "\n")
print("── Free-form sample ──")
print(json.dumps(free_sample, indent=2)[:1200], "...\n" if len(json.dumps(free_sample)) > 1200 else "\n")
if multi_sample is not free_sample:
    print(f"── Multi-blank sample ({n_ans_placeholders(multi_sample['question'])} blanks) ──")
    print(json.dumps(multi_sample, indent=2)[:1200], "...\n" if len(json.dumps(multi_sample)) > 1200 else "\n")

Loaded 1126 questions  (375 MCQ, 751 free-form) from /content/151B_SP26_Competition/data/public.jsonl

── MCQ sample ──
{
  "question": "$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $",
  "options": [
    "$0$",
    "$frac{1}{a}$",
    "$frac{3}{a}$",
    "$frac{1}{2a^2}$",
    "$frac{1}{2a}$",
    "$frac{2}{a}$",
    "$2a$",
    "$frac{3}{2a}$",
    "$frac{3}{2a^2}$",
    "$frac{1}{a^2}$"
  ],
  "answer": "F",
  "id": 1
} 

── Free-form sample ──
{
  "question": "Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]",
  "answer": [
    "325*(1+325)"
  ],
  "id": 0
} 

── Multi-blank sample (2 blanks) ──
{
  "question": "A roasted turkey is taken from an oven when its temperature has reached 185 Fahrenheit and is placed on a table in a room where the temperature is 75 Fahrenheit.\n(a) If the temperature of the turkey is 155 Fahrenheit after half an hour, what is its temperature after 45 minutes? Your answer is [ANS] Fahrenheit. (b) When will the trukey cool to 1

In [5]:
# filter to only first-pass failed questions

FIRST_PASS_PATH = Path("/content/151B_SP26_Competition/results/public.k6.judged_traces.jsonl")

failed_ids = set()

with open(FIRST_PASS_PATH, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if not r.get("has_correct", False):
            failed_ids.add(r["id"])

print("Failed questions from first pass:", len(failed_ids))

data = [d for d in data if d["id"] in failed_ids]

print("Second-pass questions:", len(data))

Failed questions from first pass: 510
Second-pass questions: 510


In [6]:
#Load judger
import sys, os
sys.path.insert(0, str(REPO_ROOT))

from judger import Judger
judger = Judger(strict_extract=False)

def judge_response(item, response):
    gold = item["answer"]
    is_mcq = bool(item.get("options"))

    if is_mcq:
        key, final_response = extract_candidate(item, response)
        pred = extract_boxed(final_response)
        pred = pred[-1] if pred else final_response
        return str(pred).strip().upper() == str(gold).strip().upper()

    gold_list = gold if isinstance(gold, list) else [gold]

    try:
        return bool(judger.auto_judge(
            pred=response,
            gold=gold_list,
            options=[[]] * len(gold_list),
        ))
    except Exception:
        return False

In [ ]:
!find /content -name "judger.py"

/content/151B_SP26_Competition/judger.py


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("/content/151B_SP26_Competition")
sys.path.insert(0, str(REPO_ROOT))

from judger import Judger
judger = Judger(strict_extract=False)

print("Judger loaded.")

Judger loaded.


## 5. Prompt construction

Per question — no global variant switch:

| Case | System prompt |
|------|----------------|
| MCQ | baseline |
| Free-form, 1 `[ANS]` | baseline |
| Free-form, 2+ `[ANS]` | multi-blank (`\boxed{a}, \boxed{b}, ...`, judger-compatible) |

In [7]:
_MATH_BASELINE = (
    "You are an expert mathematician. Solve carefully and efficiently. "
    "Be careful with arithmetic, signs, units, and exact values. "
    "Do not assume the problem contains typos. "
    "Do not modify the problem statement. "
    "Do not approximate constants unless explicitly requested. "
    "Do not overthink or explore unnecessary alternatives. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

_MCQ_BASELINE = (
    "You are an expert mathematician. "
    "Solve the problem carefully and efficiently using the problem exactly as written. "
    "Do not assume the problem contains typos. "
    "Do not modify the problem statement. "
    "Do not approximate constants unless explicitly requested. "
    "Then compare your exact result against the answer choices and choose the best matching option. "
    "Do not overthink or explore unnecessary alternatives. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

_MATH_MULTI_BLANK = (
    "You are an expert mathematician. Solve carefully and efficiently. "
    "Be careful with arithmetic, signs, units, and exact values. "
    "Do not assume the problem contains typos. "
    "Do not modify the problem statement. "
    "Do not approximate constants unless explicitly requested. "
    "Do not overthink or explore unnecessary alternatives. "
    "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
    "\\boxed{}, separated by commas, in the order the blanks appear "
    "(e.g. \\boxed{3}, \\boxed{7}). Do not use labels like 'Answer 1:' between boxes. "
    "For single-answer problems, use one \\boxed{}."
)

_MATH_BASELINE = (
    "You are an expert mathematician. Solve carefully and efficiently. "
    "Be careful with arithmetic, signs, units, and exact values. "
    "Do not assume the problem contains typos. "
    "Do not modify the problem statement. "
    "Do not approximate constants unless explicitly requested. "
    "Do not overthink or explore unnecessary alternatives. "
    "After obtaining a solution, do not reconsider it unless you find a concrete mathematical error."
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

_MCQ_BASELINE = (
    "You are an expert mathematician. "
    "Solve the problem carefully and efficiently using the problem exactly as written. "
    "First identify the problem type, the givens, and exactly what the question asks. "
    "Focus on important constraints and keywords. "
    "Do not assume the problem contains typos. "
    "Do not modify the problem statement. "
    "Do not approximate constants unless explicitly requested. "
    "Do not overthink or explore unnecessary alternatives. "
    "Then compare your exact result against the answer choices and choose the best matching option. "
    "After obtaining a solution, do not reconsider it unless you find a concrete mathematical error."
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

_MATH_MULTI_BLANK = (
    "You are an expert mathematician. Solve carefully and efficiently. "
    "Be careful with arithmetic, signs, units, and exact values. "
    "Do not assume the problem contains typos. "
    "Do not modify the problem statement. "
    "Do not approximate constants unless explicitly requested. "
    "Do not overthink or explore unnecessary alternatives. "
    "After obtaining a solution, do not reconsider it unless you find a concrete mathematical error."
    "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
    "\\boxed{}, separated by commas, in the order the blanks appear "
    "(e.g. \\boxed{3}, \\boxed{7}). Do not use labels like 'Answer 1:' between boxes. "
    "For single-answer problems, use one \\boxed{}."
)



def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return _MCQ_BASELINE, f"{question}\n\nOptions:\n{opts_text}"
    n_blanks = n_ans_placeholders(question)
    if n_blanks > 1:
        example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
        user = (
            f"{question}\n\n"
            f"The problem has {n_blanks} [ANS] blanks. "
            f"After your reasoning, give {n_blanks} comma-separated \\boxed{{}} values "
            f"in order: {example}"
        )
        return _MATH_MULTI_BLANK, user
    return _MATH_BASELINE, question


def prompt_mode(question: str, options: Optional[list]) -> str:
    if options:
        return "mcq/baseline"
    return "multi-blank" if n_ans_placeholders(question) > 1 else "baseline"


for label, item in [
    ("MCQ", mcq_sample),
    ("Free-form (single-blank)", free_sample),
    *( [(f"Multi-blank ({n_ans_placeholders(multi_sample['question'])} blanks)", multi_sample)] if multi_sample is not free_sample else [] ),
]:
    mode = prompt_mode(item["question"], item.get("options"))
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} [{mode}] ──")
    print(f"  system: {sys_p[:120]}...")
    print(f"  user (first 300 chars): {usr_p[:300]}...\n")

── MCQ [mcq/baseline] ──
  system: You are an expert mathematician. Solve the problem carefully and efficiently using the problem exactly as written. First...
  user (first 300 chars): $int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. $frac{1}{a^2}$...

── Free-form (single-blank) [baseline] ──
  system: You are an expert mathematician. Solve carefully and efficiently. Be careful with arithmetic, signs, units, and exact va...
  user (first 300 chars): Find the sum of the first $325$ positive even whole numbers. Sum: [ANS]...

── Multi-blank (2 blanks) [multi-blank] ──
  system: You are an expert mathematician. Solve carefully and efficiently. Be careful with arithmetic, signs, units, and exact va...
  user (first 300 chars): A roasted turkey is taken from an oven when its temperature has reached 185 Fahrenheit and is placed on a table in

In [8]:
def strip_thinking(text: str) -> str:
    """Keep only final text after </think>, if present."""
    text = text.strip()
    if "</think>" in text:
        return text.split("</think>")[-1].strip()
    return text


def extract_boxed(text: str) -> list[str]:
    """Extract balanced \\boxed{...} contents, including nested braces like \\frac{1}{2}."""
    text = strip_thinking(text)
    boxes = []
    token = "\\boxed"
    i = 0

    while True:
        j = text.find(token, i)
        if j == -1:
            break

        k = j + len(token)
        while k < len(text) and text[k].isspace():
            k += 1

        if k >= len(text) or text[k] != "{":
            i = k
            continue

        depth = 0
        start = k + 1
        end = None

        for p in range(k, len(text)):
            if text[p] == "{":
                depth += 1
            elif text[p] == "}":
                depth -= 1
                if depth == 0:
                    end = p
                    break

        if end is None:
            break

        boxes.append(text[start:end].strip())
        i = end + 1

    return boxes


def normalize_answer(ans: str) -> str:
    """Normalize equivalent-looking final answers for voting."""
    if ans is None:
        return ""

    s = str(ans).strip()
    s = s.strip("$")
    s = s.replace("−", "-")
    s = s.replace("\\left", "").replace("\\right", "")
    s = s.replace("\\,", "").replace("\\!", "")
    s = s.replace("\\dfrac", "\\frac").replace("\\tfrac", "\\frac")
    s = re.sub(r"\\text\{([^{}]*)\}", r"\1", s)
    s = re.sub(r"\s+", "", s)
    s = s.rstrip(".")

    # Normalize simple LaTeX fractions.
    m = re.fullmatch(r"\\frac\{(-?\d+)\}\{(-?\d+)\}", s)
    if m:
        num = int(m.group(1))
        den = int(m.group(2))
        if den != 0:
            return str(Fraction(num, den))

    # Normalize plain fractions and decimals.
    try:
        if re.fullmatch(r"-?\d+/\d+", s):
            return str(Fraction(s))
        if re.fullmatch(r"-?\d+(\.\d+)?", s):
            return str(Fraction(s))
    except Exception:
        pass

    return s.lower()


def format_question_with_options(item: dict) -> str:
    question = item["question"]
    options = item.get("options")

    if not options:
        return question

    labels = [chr(65 + i) for i in range(len(options))]
    opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
    return f"{question}\n\nOptions:\n{opts_text}"


def extract_candidate(item: dict, response: str) -> tuple[tuple, str]:
    """
    Returns:
      key: normalized key used for voting
      final_response: clean boxed answer for CSV
    """
    boxes = extract_boxed(response)
    options = item.get("options")

    # MCQ handling
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        candidate = boxes[-1].strip() if boxes else strip_thinking(response).strip()
        candidate_clean = candidate.upper().strip().replace(".", "")

        if candidate_clean in labels:
            return ("mcq", candidate_clean), f"\\boxed{{{candidate_clean}}}"

        # If the model outputs the choice text instead of the letter, map it back.
        norm_candidate = normalize_answer(candidate)
        for label, opt in zip(labels, options):
            if norm_candidate == normalize_answer(opt):
                return ("mcq", label), f"\\boxed{{{label}}}"

        return ("invalid_mcq", normalize_answer(candidate)), f"\\boxed{{{candidate}}}"

    # Free-response handling
    n_blanks = n_ans_placeholders(item["question"])

    if n_blanks > 1:
        if len(boxes) >= n_blanks:
            vals = boxes[-n_blanks:]
        elif len(boxes) == 1 and "," in boxes[-1]:
            vals = [x.strip() for x in boxes[-1].split(",")]
        else:
            vals = boxes if boxes else [strip_thinking(response).strip()]

        key = tuple(normalize_answer(v) for v in vals)
        final_response = ", ".join(f"\\boxed{{{v}}}" for v in vals)
        return ("multi", key), final_response

    # Single free-response
    if boxes:
        val = boxes[-1]
    else:
        lines = [x.strip() for x in strip_thinking(response).splitlines() if x.strip()]
        val = lines[-1] if lines else strip_thinking(response).strip()

    return ("free", normalize_answer(val)), f"\\boxed{{{val}}}"


def vote_responses(item: dict, raw_responses: list[str]) -> dict:
    records = []

    for raw in raw_responses:
        key, final_response = extract_candidate(item, raw)
        records.append({
            "key": key,
            "final_response": final_response,
            "raw_response": raw,
        })

    counts = Counter(r["key"] for r in records)
    best_key, best_count = counts.most_common(1)[0]

    # Pick the first raw response that produced the winning normalized answer.
    best_record = next(r for r in records if r["key"] == best_key)

    return {
        "key": best_key,
        "agreement": best_count / len(raw_responses),
        "final_response": best_record["final_response"],
        "raw_response": best_record["raw_response"],
        "all_keys": [r["key"] for r in records],
    }


def final_format_bad(item: dict, final_response: str) -> bool:
    boxes = extract_boxed(final_response)
    options = item.get("options")

    if options:
        if len(boxes) != 1:
            return True
        labels = [chr(65 + i) for i in range(len(options))]
        return boxes[0].strip().upper() not in labels

    n_blanks = n_ans_placeholders(item["question"])
    if n_blanks > 1:
        return len(boxes) != n_blanks

    return len(boxes) != 1


def should_verify(item: dict, vote: dict) -> bool:
    if not USE_VERIFIER:
        return False

    if vote["agreement"] < VERIFY_AGREEMENT_THRESHOLD:
        return True

    if str(vote["key"][0]).startswith("invalid"):
        return True

    if final_format_bad(item, vote["final_response"]):
        return True

    return False


def build_verifier_prompt(item: dict, proposed_answer: str) -> tuple[str, str]:
    system = (
        "You are a strict math verifier. Check the proposed answer carefully. "
        "If it is correct, repeat it. If it is wrong, correct it. "
        "Return only the final answer in the required boxed format."
    )

    n_blanks = n_ans_placeholders(item["question"])
    if item.get("options"):
        required = "Return exactly one option letter inside \\boxed{}, e.g. \\boxed{C}."
    elif n_blanks > 1:
        example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
        required = f"Return exactly {n_blanks} boxed answers in order: {example}."
    else:
        required = "Return exactly one answer inside \\boxed{}."

    user = (
        f"Problem:\n{format_question_with_options(item)}\n\n"
        f"Proposed answer:\n{proposed_answer}\n\n"
        f"{required}"
    )

    return system, user

## 6. Load model with vLLM (A100 profile)

Same **bf16** profile as `full_public.ipynb` — not the starter L4 INT8 path. See [`docs/infra/vllm-inference-config.md`](../docs/infra/vllm-inference-config.md).

| Parameter | Value |
|-----------|-------|
| `dtype` | `bfloat16` |
| `max_model_len` | 32768 |
| `enable_prefix_caching` | True |
| `enable_chunked_prefill` | True |

In [9]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

with _jupyter_stdout_for_vllm():
    llm = LLM(
        model=MODEL_ID,
        dtype="bfloat16",
        enable_prefix_caching=True,
        gpu_memory_utilization=0.92,
        max_model_len=15000,
        trust_remote_code=True,
        max_num_seqs=128,
        max_num_batched_tokens=15000,
        enable_chunked_prefill=True,
    )
answer_sampling_params = SamplingParams(
    n=SELF_CONSISTENCY_K,
    max_tokens=MAX_TOKENS,
    temperature=0.7,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
    seed=None,
)

verifier_sampling_params = SamplingParams(
    n=1,
    max_tokens=4096,
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
    seed=123,
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

INFO 05-31 19:42:34 [cuda.py:368] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
INFO 05-31 19:42:34 [flash_attn.py:646] Using FlashAttention version 2


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

INFO 05-31 19:42:51 [weight_utils.py:615] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 15.924894 seconds
INFO 05-31 19:42:51 [weight_utils.py:904] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 7.49 GiB. Available RAM: 169.19 GiB.
INFO 05-31 19:42:51 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-31 19:42:52 [default_loader.py:384] Loading weights took 0.99 seconds
INFO 05-31 19:42:53 [gpu_model_runner.py:4879] Model loading took 7.61 GiB memory and 17.770547 seconds
INFO 05-31 19:43:02 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/28fccb4ed3/rank_0_0/backbone for vLLM's torch.compile
INFO 05-31 19:43:02 [backends.py:1128] Dynamo bytecode transform time: 8.91 s
INFO 05-31 19:43:07 [backends.py:376] Cache the graph of compile range (1, 15000) for later use
INFO 05-31 19:43:10 [backends.py:391] Compiling a graph for compile range (1, 15000) takes 8.46 s
INFO 05-31 19:43:13 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/eb1dc5c0b0df9dc1842b7fe34b8fdd39f2160858090b18171cf8ac60359606f8/rank_0_0/model
INFO 05-31 19:43:13 [monitor.py:53] torch.compile took 20.05 s in total
INFO 05-31 19:43:13 [monitor.py:81] Initial profiling/warmup run took 0.07 s
INFO 05-31 19:43:18 [gpu_model_runne

In [10]:
print("Failed questions from first pass:", len(failed_ids))
print("Second-pass questions:", len(data))

Failed questions from first pass: 510
Second-pass questions: 510


## 7. Generate responses

Checkpoint: `results/submission.responses.jsonl` (Drive on Colab). Delete checkpoint to regenerate from scratch.

In [11]:


CHUNK_SIZE = max(4, 64 // SELF_CONSISTENCY_K)

_results_dir = Path(SUBMISSION_CSV).parent
_results_dir.mkdir(parents=True, exist_ok=True)

response_checkpoint = _results_dir / "public.failed.k4.12k.judged_traces.jsonl"
sft_trace_path = _results_dir / "public.failed.k4.12k.correct_sft_traces.jsonl"

#if response_checkpoint.exists():
#    response_checkpoint.unlink()

#if sft_trace_path.exists():
#    sft_trace_path.unlink()

completed_records = {}

if response_checkpoint.exists():
    with open(response_checkpoint, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            completed_records[r["id"]] = r
    print(f"Resumed: {len(completed_records)} questions already processed")

remaining = [d for d in data if d["id"] not in completed_records]
print(f"Generating {len(remaining)} remaining / {len(data)} total")

for chunk_start in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[chunk_start:chunk_start + CHUNK_SIZE]

    prompts = []
    for item in chunk:
        system, user = build_prompt(item["question"], item.get("options"))
        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append(prompt_text)

    with _jupyter_stdout_for_vllm():
        outputs = llm.generate(prompts, sampling_params=answer_sampling_params)

    records = []

    for item, out in zip(chunk, outputs):
        raw_samples = [sample.text.strip() for sample in out.outputs]

        judged_samples = []
        correct_samples = []

        for raw in raw_samples:
            is_correct = judge_response(item, raw)
            key, final_response = extract_candidate(item, raw)

            sample_record = {
                "response": raw,
                "final_response": final_response,
                "vote_key": repr(key),
                "correct": is_correct,
            }

            judged_samples.append(sample_record)

            if is_correct:
                correct_samples.append(sample_record)

        record = {
            "id": item["id"],
            "question": item["question"],
            "answer": item["answer"],
            "num_samples": len(raw_samples),
            "num_correct": len(correct_samples),
            "has_correct": len(correct_samples) > 0,
            "judged_samples": judged_samples,
            "correct_samples": correct_samples,
        }

        records.append(record)

    with open(response_checkpoint, "a", encoding="utf-8") as f:
        for record in records:
            completed_records[record["id"]] = record
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    done = len(completed_records)
    total_attempts = sum(r["num_samples"] for r in completed_records.values())
    correct_attempts = sum(r["num_correct"] for r in completed_records.values())
    covered_questions = sum(r["has_correct"] for r in completed_records.values())

    print(
        f"{done}/{len(data)} questions | "
        f"attempt accuracy: {correct_attempts / total_attempts * 100:.2f}% | "
        f"question coverage: {covered_questions / done * 100:.2f}%"
    )

SYSTEM_PROMPT = (
    "You are an expert mathematician. Solve carefully but efficiently. "
    "Do not assume the problem has typos. Do not invent missing terms. "
    "Do not approximate constants unless the question explicitly asks. "
    "For multiple-choice questions, choose the option that best matches the exact result. "
    "End with exactly one final answer in the format \\boxed{...}."
)

with open(sft_trace_path, "w", encoding="utf-8") as f:
    for record in completed_records.values():
        for s in record["correct_samples"]:
            sft_record = {
                "id": record["id"],
                "messages": [
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT,
                    },
                    {
                        "role": "user",
                        "content": record["question"],
                    },
                    {
                        "role": "assistant",
                        "content": s["response"],
                    },
                ],
                #"answer": record["answer"],
            }

            f.write(json.dumps(sft_record, ensure_ascii=False) + "\n")

total_attempts = sum(r["num_samples"] for r in completed_records.values())
correct_attempts = sum(r["num_correct"] for r in completed_records.values())
covered_questions = sum(r["has_correct"] for r in completed_records.values())

print("=" * 50)
print(f"Questions processed: {len(completed_records)}")
print(f"Total attempts: {total_attempts}")
print(f"Correct attempts: {correct_attempts}")
print(f"Attempt-level accuracy: {correct_attempts / total_attempts * 100:.2f}%")
print(f"Questions with >=1 correct sample: {covered_questions}")
print(f"Question-level coverage: {covered_questions / len(completed_records) * 100:.2f}%")
print(f"Correct SFT traces saved to: {sft_trace_path}")
print("=" * 50)

Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Generating 510 remaining / 510 total


Rendering prompts:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

480/510 questions | attempt accuracy: 18.80% | question coverage: 26.67%


Rendering prompts:   0%|          | 0/14 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/56 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

496/510 questions | attempt accuracy: 18.80% | question coverage: 26.81%
510/510 questions | attempt accuracy: 18.77% | question coverage: 26.67%
Questions processed: 510
Total attempts: 2040
Correct attempts: 383
Attempt-level accuracy: 18.77%
Questions with >=1 correct sample: 136
Question-level coverage: 26.67%
Correct SFT traces saved to: /content/151B_SP26_Competition/results/public.failed.k4.12k.correct_sft_traces.jsonl


In [12]:
truncated = 0
total = 0
no_box = 0

for r in completed_records.values():
    for s in r["judged_samples"]:
        resp = s["response"]
        total += 1

        if "\\boxed" not in resp:
            no_box += 1

        # crude truncation signal: response ends without final punctuation/box
        if "\\boxed" not in resp or resp.strip().endswith(("=", "+", "-", "*", "/", "to", "is")):
            truncated += 1

print("Total samples:", total)
print("No boxed answer:", no_box, f"({no_box/total*100:.2f}%)")
print("Possible truncation:", truncated, f"({truncated/total*100:.2f}%)")

Total samples: 2040
No boxed answer: 542 (26.57%)
Possible truncation: 551 (27.01%)


In [15]:
# Merge both files
import json

records = []

for path in [
    "/content/151B_SP26_Competition/results/public.k6.judged_traces.jsonl",
    "/content/151B_SP26_Competition/results/public.failed.k4.12k.judged_traces.jsonl",
]:
    with open(path, encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

print("records:", len(records))

records: 1636


In [21]:
import random

random.seed(42)

MAX_PAIRS_PER_QUESTION = 3
MIN_CHOSEN_LEN = 50
MIN_REJECTED_LEN = 20

dpo_examples = []

for record in records:
    corrects = [
        s for s in record["judged_samples"]
        if (
            s["correct"]
            and len(s["response"]) > MIN_CHOSEN_LEN
        )
    ]

    # Prefer boxed wrongs, but allow non-boxed wrongs if needed
    boxed_wrongs = [
        s for s in record["judged_samples"]
        if (
            not s["correct"]
            and "\\boxed" in s["response"]
            and len(s["response"]) > MIN_REJECTED_LEN
        )
    ]

    all_wrongs = [
        s for s in record["judged_samples"]
        if (
            not s["correct"]
            and len(s["response"]) > MIN_REJECTED_LEN
        )
    ]

    wrongs = boxed_wrongs if boxed_wrongs else all_wrongs

    if not corrects or not wrongs:
        continue

    pairs = []
    for chosen in corrects:
        for rejected in wrongs:
            pairs.append((chosen, rejected))

    random.shuffle(pairs)
    pairs = pairs[:MAX_PAIRS_PER_QUESTION]

    for chosen, rejected in pairs:
        dpo_examples.append({
            "prompt": record["question"],
            "chosen": chosen["response"],
            "rejected": rejected["response"],
        })

print("DPO pairs:", len(dpo_examples))

DPO pairs: 803


In [22]:
total_records = 0
has_correct = 0
has_wrong = 0
has_boxed_wrong = 0
has_pair_strict = 0

for record in records:
    total_records += 1

    corrects_all = [s for s in record["judged_samples"] if s["correct"]]
    wrongs_all = [s for s in record["judged_samples"] if not s["correct"]]
    wrongs_boxed = [s for s in wrongs_all if "\\boxed" in s["response"]]

    corrects_strict = [
        s for s in corrects_all
        if len(s["response"]) > 200
    ]

    wrongs_strict = [
        s for s in wrongs_all
        if "\\boxed" in s["response"] and len(s["response"]) > 200
    ]

    if corrects_all:
        has_correct += 1
    if wrongs_all:
        has_wrong += 1
    if wrongs_boxed:
        has_boxed_wrong += 1
    if corrects_strict and wrongs_strict:
        has_pair_strict += 1

print("records:", total_records)
print("records with correct:", has_correct)
print("records with wrong:", has_wrong)
print("records with boxed wrong:", has_boxed_wrong)
print("records with strict pair:", has_pair_strict)

records: 1636
records with correct: 752
records with wrong: 1159
records with boxed wrong: 652
records with strict pair: 117


In [ ]:
# check pairs

print("DPO pairs:", len(dpo_examples))

for i in range(3):
    print("=" * 80)
    print("PROMPT")
    print(dpo_examples[i]["prompt"][:300])

    print("\nCHOSEN")
    print(dpo_examples[i]["chosen"][:500])

    print("\nREJECTED")
    print(dpo_examples[i]["rejected"][:500])


In [24]:
# Save DPO file

import json
from datasets import Dataset


DPO_PATH = "/content/151B_SP26_Competition/results/public_dpo_803_pairs.jsonl"

with open(DPO_PATH, "w", encoding="utf-8") as f:
    for ex in dpo_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print("Saved:", DPO_PATH)

Saved: /content/151B_SP26_Competition/results/public_dpo_803_pairs.jsonl


In [2]:
# create dataset
import json
from datasets import Dataset


DPO_PATH = "/content/151B_SP26_Competition/results/public_dpo_803_pairs.jsonl"

rows = []

with open(DPO_PATH, encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

dataset = Dataset.from_list(rows)

print(dataset)
print(dataset[0].keys())

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 803
})
dict_keys(['prompt', 'chosen', 'rejected'])


In [4]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

SYSTEM_PROMPT = (
    "You are an expert mathematician. "
    "Solve carefully and efficiently. "
    "Do not assume the problem contains typos. "
    "Do not modify the problem statement. "
    "Do not approximate constants unless explicitly requested. "
    "Do not overthink or explore unnecessary alternatives. "
    "After obtaining a solution, do not reconsider it unless you find a concrete mathematical error. "
    "Put your final answer inside \\boxed{}."
)

def format_dpo(ex):
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": ex["prompt"]},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

    return {
        "prompt": prompt,
        "chosen": ex["chosen"],
        "rejected": ex["rejected"],
    }



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
# Format dataset

dataset = dataset.map(
    format_dpo,
    remove_columns=dataset.column_names,
)

print(dataset)
print(dataset[0].keys())
print(dataset[0]["prompt"][:300])
print("=" * 80)
print(dataset[0]["chosen"][:500])
print("=" * 80)
print(dataset[0]["rejected"][:500])

Map:   0%|          | 0/803 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 803
})
dict_keys(['prompt', 'chosen', 'rejected'])
<|im_start|>system
You are an expert mathematician. Solve carefully and efficiently. Do not assume the problem contains typos. Do not modify the problem statement. Do not approximate constants unless explicitly requested. Do not overthink or explore unnecessary alternatives. After obtaining a soluti
Okay, let's try to solve this problem step by step. The question is asking for the analytic function \( f(z) = u + iv \) where \( u(x, y) = x^3 + 6x^2y - 3xy^2 - 2y^3 \), and \( f(0) = 0 \). 

First, I remember that for a function \( f(z) = u + iv \) to be analytic, it must satisfy the Cauchy-Riemann equations: \( u_x = v_y \) and \( u_y = -v_x \). Also, since \( u \) is given, we can find \( v \) by integrating the Cauchy-Riemann equations. 

Let me start by computing the partial derivatives of
This is a complex or challenging question, and it is difficult to provide a

In [30]:
%pip install -q "trl>=0.18" "peft>=0.15" accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 70.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.2.1 which is incompatible.


In [6]:
# Load model with QLoRA
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [33]:
import trl
print(trl.__version__)

1.5.1


In [37]:
!zip public_dpo_803_pairs.zip \
/content/151B_SP26_Competition/results/public_dpo_803_pairs.jsonl

  adding: content/151B_SP26_Competition/results/public_dpo_803_pairs.jsonl (deflated 80%)


In [38]:
from google.colab import files
files.download("public_dpo_803_pairs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# Train DPO
training_args = DPOConfig(
    output_dir="/content/qwen3_dpo_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-7,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    bf16=True,
    report_to="none",
    beta=0.1,
)

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/803 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/803 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.698517
20,0.697014
30,0.695793
40,0.697148
50,0.729523
60,0.696467
70,0.696765
80,0.685392
90,0.696424
100,0.706912


TrainOutput(global_step=101, training_loss=0.6993926987789645, metrics={'train_runtime': 357.2629, 'train_samples_per_second': 2.248, 'train_steps_per_second': 0.283, 'total_flos': 3.597233017356288e+16, 'train_loss': 0.6993926987789645, 'entropy': 0.22824802994728088, 'num_tokens': 1641916.0, 'logits/chosen': -4.290437454384759, 'logits/rejected': -3.181199513045478, 'mean_token_accuracy': 0.9332895874977112, 'rewards/chosen': 0.1461311342815558, 'rewards/rejected': 0.033750917141636215, 'rewards/accuracies': 1.0, 'rewards/margins': 0.11238021527727445, 'logps/chosen': -150.3252741495768, 'logps/rejected': -216.56288146972656, 'epoch': 1.0})

In [8]:
SAVE_DIR = "/content/qwen3_dpo_lora_final"

trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved:", SAVE_DIR)

Saved: /content/qwen3_dpo_lora_final


In [9]:
!zip -r qwen3_dpo_lora_final.zip /content/qwen3_dpo_lora_final

  adding: content/qwen3_dpo_lora_final/ (stored 0%)
  adding: content/qwen3_dpo_lora_final/README.md (deflated 65%)
  adding: content/qwen3_dpo_lora_final/tokenizer.json (deflated 81%)
  adding: content/qwen3_dpo_lora_final/chat_template.jinja (deflated 76%)
  adding: content/qwen3_dpo_lora_final/adapter_model.safetensors (deflated 21%)
  adding: content/qwen3_dpo_lora_final/tokenizer_config.json (deflated 60%)
  adding: content/qwen3_dpo_lora_final/adapter_config.json (deflated 59%)


## 8. Write submission CSV

Uses Python’s `csv` writer so commas and newlines inside `response` are quoted per RFC 4180.

In [ ]:
try:
    csv_out = DRIVE_PROJECT_ROOT / "results" / Path(SUBMISSION_CSV).name
except NameError:
    csv_out = Path(SUBMISSION_CSV)

csv_out.parent.mkdir(parents=True, exist_ok=True)

with open(csv_out, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
    w.writerow(["id", "response"])
    for row in data:
        qid = row["id"]
        w.writerow([qid, completed_records[qid]["response"]])

print(f"Wrote {len(data)} rows to {csv_out.resolve()}")

with open(csv_out, newline="", encoding="utf-8") as f:
    n = sum(1 for _ in f)
assert n == len(data) + 1, f"Expected header + {len(data)} rows, got {n} lines"
print("CSV line count OK (1 header +", len(data), "data rows).")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil

RESULT_DIR = "/content/151B_SP26_Competition/results"

shutil.copy(
    f"{RESULT_DIR}/public.failed.k4.12k.correct_sft_traces.jsonl",
    "/content/drive/MyDrive/public.failed.k4.12k.correct_sft_traces.jsonl"
)

shutil.copy(
    f"{RESULT_DIR}/public.failed.k4.12k.judged_traces.jsonl",
    "/content/drive/MyDrive/public.failed.k4.12k.judged_traces.jsonl"
)

print("Saved to Drive")

Saved to Drive


In [ ]:
!ls -lh /content/151B_SP26_Competition/results

total 162M
-rw-r--r-- 1 root root  23M May 31 06:04 public.k6.correct_sft_traces.jsonl
-rw-r--r-- 1 root root 140M May 31 06:04 public.k6.judged_traces.jsonl


In [ ]:
count = 0
for r in completed_records.values():
    for s in r["judged_samples"]:
        if not s["correct"]:
            print(s["response"][-1000:])
            print("="*100)
            count += 1
            if count >= 10:
                raise SystemExit

m assumes a **numerical approximation** (e.g., using $ \pi \approx 3 $), then $ \frac{\pi}{a} \approx \frac{3}{a} $, and if the problem also **omits the factor of 2** (as in some engineering contexts), it might be interpreted as $ \frac{1}{2a} $.

Among the given options, **option E** is $ \frac{1}{2a} $, which is the **simplest and most plausible** choice that could arise from a common misinterpretation or approximation of the standard result $ \frac{\pi}{a} $, especially if the problem was meant to be from 0 to $ \infty $ (where the integral is $ \frac{\pi}{2a} $).

---

### Final Reasoning

Given the lack of $ \pi $ in the options and the context of multiple-choice questions in applied mathematics, it's reasonable to infer that the problem was intended to ask for a simpler version of the integral (e.g., from 0 to $ \infty $), and the answer was approximated or simplified.

The **most likely intended answer**, based on common misinterpretations or approximations, is:

$$
\boxed{E}
$$

SystemExit: 

## Done

Upload **`submission.csv`** to the course leaderboard workflow. Keep **`submission.responses.jsonl`** as a backup of raw traces.

Pipeline matches **pub-003** (`full_public.ipynb`): 32k tokens, bf16 A100, adaptive multi-blank prompts.

In [ ]:
try:
    from google.colab import runtime
    drive.flush_and_unmount()
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")
except NameError:
    print("Drive not mounted — skipping runtime termination.")

In [ ]:
!ls

sample_data
